# Inference
This notebook provides easy inference code for the models implemented under `model/`.
It covers:
- Reconstruction from the FREUD model
- Forecasting with the RaMViD Latent Space Model (LSM)

## 1. FREUD Reconstructions

In [1]:
from data.sevir import SevirDataModule

dmodule = SevirDataModule(
    sevir_pth = "/export/group/datasets/SEVIR",  # TODO: change to local data path
    seq_len = 25,
    normalize = True,
    batch_size = 1,
    num_workers = 8,
    val_files_index = "data/test_data.txt",
    use_duplicate_validation_seq = True,
    seed = 42,
)

train_loader = dmodule.train_dataloader()
val_loader = dmodule.val_dataloader()

for batch in val_loader:
    print(batch.keys())
    break

dict_keys(['x', 'metadata'])


In [2]:
import torch
import mediapy as mp
import einops
from jaxtyping import Float
from model.freud import FreudDiffusionAE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

def show_video(x, title=None):
    vid = x.float().cpu().numpy()
    vid = einops.repeat(vid, "t h w 1 -> t h w (1 three)", three=3)
    vid = (vid + 1) * 127.5
    vid = vid.astype('uint8')
    mp.show_video(vid, fps=5, title=title)

@torch.inference_mode()
def get_recon(
    model: FreudDiffusionAE,
    x: Float[torch.Tensor, "B T H W C"],
    decoding_sample_steps: int=10,
) -> Float[torch.Tensor, "B T H W C"]:
    with torch.autocast(device_type=DEVICE.type, dtype=DTYPE):
        latent = model.encode(x)
        recon = model.decode(latent, sample_steps=decoding_sample_steps)
    return recon, latent

In [3]:
from model.freud import get_freud_model

model = get_freud_model(ckpt_path="ckpt/freud_ckpt.pt")
model = model.to(device=DEVICE, dtype=DTYPE)
model.eval()

FreudDiffusionAE(
  (unet): FreudDecoder(
    (sub_modules): ModuleList(
      (0): FreudDecoderLevel(
        (token_merge): TokenMerge3D(
          (proj): Linear(in_features=16, out_features=96, bias=False)
        )
        (factorized_transformer_layers): ModuleList(
          (0-1): 2 x FactorizedAttentionLayer(
            (self_attn): LegacyFactorizedAttentionBlock(
              (spatial_norm): AdaRMSNorm(
                eps=1e-06,
                (linear): Linear(in_features=384, out_features=96, bias=False)
              )
              (temporal_norm): AdaRMSNorm(
                eps=1e-06,
                (linear): Linear(in_features=384, out_features=96, bias=False)
              )
              (spatial_qkv_proj): Linear(in_features=96, out_features=288, bias=False)
              (temporal_qkv_proj): Linear(in_features=96, out_features=288, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
              (spatial_out_proj): Linear(in_features=96, out_fea

In [4]:
x = batch['x'].to(device=DEVICE, dtype=DTYPE)
recon, latent = get_recon(model, x)
print(f"Input x shape: {x.shape}, recon shape: {recon.shape}, latent shape: {latent.shape}")

recon_ = recon.float().clamp(-1, 1)

to_show = torch.cat([x, recon_, recon_ - x], dim=3)
show_video(to_show[0], title="Input | Recon | Diff")

Input x shape: torch.Size([1, 25, 384, 384, 1]), recon shape: torch.Size([1, 25, 384, 384, 1]), latent shape: torch.Size([1, 25, 48, 48, 4])


Input | Recon | Diff This browser does not support the video tag.


## 2. LSM Forecasts

In [5]:
from data.sevir import SevirDataModule

dmodule = SevirDataModule(
    sevir_pth = "/export/group/datasets/SEVIR",  # TODO: change to local data path
    seq_len = 25,
    normalize = True,
    batch_size = 1,
    num_workers = 8,
    val_files_index = "data/test_data.txt",
    use_duplicate_validation_seq = True,
    seed = 42,
)

train_loader = dmodule.train_dataloader()
val_loader = dmodule.val_dataloader()

for batch in val_loader:
    print(batch.keys())
    break

dict_keys(['x', 'metadata'])


In [6]:
import torch
import mediapy as mp
import einops
from jaxtyping import Float
from model.lsm import RaMViDFM

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

def show_video(x, title=None):
    vid = x.float().cpu().numpy()
    vid = einops.repeat(vid, "t h w 1 -> t h w (1 three)", three=3)
    vid = (vid + 1) * 127.5
    vid = vid.astype('uint8')
    mp.show_video(vid, fps=5, title=title)

@torch.inference_mode()
def get_pred(
    model: RaMViDFM,
    sample: Float[torch.Tensor, "B T H W C"],
    cond_indices: Float[torch.Tensor, "B T"],
    sample_steps: int = 25,
    decoder_sample_steps: int = 10,
    device: torch.device = torch.device("cuda:0"),
    dtype: torch.dtype = torch.bfloat16,
    decode_chunk_size: int = 4,
) -> Float[torch.Tensor, "B T H W C"]:
    cond_indices = cond_indices.bool()
    cond_idx_exp = cond_indices[:, :, None, None, None]
    condition = torch.where(cond_idx_exp, sample, torch.zeros_like(sample))
    
    with torch.autocast(device_type=device.type, dtype=dtype):
        latent_condition = model.encode(condition, force_encode=True)
        latent_noise = torch.randn_like(latent_condition)
        pred = model.sample(
            data=latent_condition,
            noise=latent_noise,
            conditional_indices=cond_indices,
            sample_steps=sample_steps,
            decode_sample_steps=decoder_sample_steps,
            data_is_latent=True,
            return_decoded=False,
        )
        pred = torch.cat(
            [
                model.decode(pred[start:start + decode_chunk_size], sample_steps=decoder_sample_steps)
                for start in range(0, pred.shape[0], decode_chunk_size)
            ],
            dim=0,
        )
    pred = pred.float().clamp(-1, 1)
    return pred

In [7]:
from model.lsm import get_lsm

model = get_lsm(ckpt_path="ckpt/lsm_ckpt.pt")
model = model.to(device=DEVICE, dtype=DTYPE)
model.eval()

RaMViDFM(
  (backbone): LSMSiT(
    (sub_modules): ModuleList(
      (0): LSMLevel(
        (token_merge): TokenMerge3D(
          (proj): Linear(in_features=16, out_features=1024, bias=False)
        )
        (factorized_transformer_layers): ModuleList(
          (0-23): 24 x FactorizedAttentionLayer(
            (self_attn): FactorizedAttentionBlock(
              (norm): AdaRMSNorm(
                eps=1e-06,
                (linear): Linear(in_features=512, out_features=1024, bias=False)
              )
              (spatial_qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
              (temporal_qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
              (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
              (spatial_pos_emb): AxialRoPE2D(dim=32, n_heads=16)
              (temporal_pos_emb): AxialRoPE1D(dim=32, n_heads=16)
            )
            (ff): FeedF

In [8]:
INPUT_LENGTH = 12

x = batch['x']
sample = x.clone().to(device=DEVICE, dtype=DTYPE)
noise = torch.randn_like(sample)
sample[:, INPUT_LENGTH:] = noise[:, INPUT_LENGTH:]
cond_indices = torch.zeros(sample.shape[0], sample.shape[1], dtype=torch.bool, device=sample.device)
cond_indices[:, :INPUT_LENGTH] = True
pred = get_pred(model, sample, cond_indices, sample_steps=25, decoder_sample_steps=10, device=DEVICE, dtype=DTYPE)
print(f"Sample shape: {sample.shape}, pred shape: {pred.shape}")

Sample shape: torch.Size([1, 25, 384, 384, 1]), pred shape: torch.Size([1, 25, 384, 384, 1])


In [9]:
pred_ = pred.float().cpu().clamp(-1, 1)

to_show = torch.cat([x, pred_, pred_ - x], dim=3)
show_video(to_show[0], title="Input | Prediction | Diff")

Input | Prediction | Diff This browser does not support the video tag.
